# 🏗️ Assistente para Edifícios Eficientes quanto a Água e Energia
## Fine-Tuning de LLM para Edificações Verdes e Net Zero

**Disciplina**: LLM e Fine-Tuning  
**Objetivo**: Chatbot especialista em LEED, AQUA-HQE, GBC Brasil EDGE, energia solar, reuso de água e NZEBs  
**Stack**: Llama 3.2 3B · QLoRA · ChromaDB · multilingual-e5-large · Gradio  
**Ambiente**: Google Colab T4 GPU

---
### ⚠️ Antes de começar
1. Ative a GPU: *Runtime → Change runtime type → T4 GPU*
2. Conecte o Google Drive (opcional, para persistência entre sessões)
3. Execute as células em ordem; checkpoints são salvos a cada etapa


## 🔧 Instalação de Dependências

In [ ]:
# Montar Google Drive para acessar os arquivos do projeto
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

# Caminho no Drive (ajuste se você nomeou a pasta diferente)
DRIVE_PATH = Path("/content/drive/MyDrive/edificios_verdes/projeto_edificios_verdes")

# Copiar para o diretório de trabalho do Colab
WORK_DIR = Path("/content/projeto_edificios_verdes")

if not WORK_DIR.exists():
    shutil.copytree(str(DRIVE_PATH), str(WORK_DIR))
    print(f"✅ Projeto copiado para {WORK_DIR}")
else:
    print(f"✅ Projeto já existe em {WORK_DIR}")

# Confirmar arquivos
print("\nCorpus encontrado:")
for f in sorted((WORK_DIR / "corpus").glob("*.txt")):
    print(f"  {f.name}")

print(f"\nPares Q&A: {sum(1 for _ in open(WORK_DIR / 'data/qa_pairs.jsonl'))} pares")

Mounted at /content/drive
✅ Projeto copiado para /content/projeto_edificios_verdes

Corpus encontrado:
  doc_01_leed_eficiencia_hidrica.txt
  doc_02_leed_energia_atmosfera.txt
  doc_03_aqua_hqe_referencial.txt
  doc_04_gbc_brasil_edge.txt
  doc_05_nbr15575_desempenho.txt
  doc_06_sistemas_fotovoltaicos.txt
  doc_07_aguas_cinzas_reuso.txt
  doc_08_bems_automacao.txt
  doc_09_procel_edifica.txt
  doc_10_net_zero_iea.txt

Pares Q&A: 492 pares


In [ ]:
import subprocess, sys, os

# ── Passo 1: remover pacotes conflitantes ────────────────────
# Necessário para evitar PicklingError (trl) e numpy ABI mismatch
print("🧹 Removendo versões conflitantes...")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall",
     "trl", "transformers", "numpy", "datasets", "-y", "-q"],
    check=False
)

# ── Passo 2: instalar versões pinadas e compatíveis ──────────
pkgs = [
    # numpy ANTES de tudo — evita ABI mismatch com torch
    "numpy==1.26.4",
    # transformers compatível com trl 0.9.6 e unsloth
    "transformers==4.46.3",
    # trl 0.9.6 — primeira versão estável com SFTConfig
    "trl==0.9.6",
    # unsloth por último — ajusta suas dependências internas
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "chromadb",
    "sentence-transformers",
    "evaluate",
    "rouge-score",
    "bert-score",
    "gradio>=4.0",
    "datasets",
    "nltk",
]

for pkg in pkgs:
    print(f"  ⏳ instalando {pkg.split('@')[0].split('==')[0].split('>=')[0]}...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("\n✅ Instalação concluída!")
print("⚠️  REINICIANDO KERNEL automaticamente...")
print("   Após o restart, execute as células a partir da célula de imports (não reinstale).")
os.kill(os.getpid(), 9)  # Restart obrigatório — limpa binários cached em memória


🧹 Removendo versões conflitantes...
  ⏳ instalando numpy...
  ⏳ instalando transformers...
  ⏳ instalando trl...
  ⏳ instalando unsloth[colab-new] ...
  ⏳ instalando chromadb...
  ⏳ instalando sentence-transformers...
  ⏳ instalando evaluate...
  ⏳ instalando rouge-score...
  ⏳ instalando bert-score...
  ⏳ instalando gradio...
  ⏳ instalando datasets...
  ⏳ instalando nltk...


## 📦 Imports e Configuração Global

In [ ]:
import os, json, re, time, warnings
from pathlib import Path
import torch

warnings.filterwarnings("ignore")

# ── Caminhos do projeto ──────────────────────────────────────
# Usa /content/ se disponível (Colab), senão ./  (local)
BASE_DIR    = Path("/content/projeto_edificios_verdes") \
              if Path("/content").exists() \
              else Path("./projeto_edificios_verdes")

CORPUS_DIR  = BASE_DIR / "corpus"
DATA_DIR    = BASE_DIR / "data"
CHROMA_DIR  = BASE_DIR / "chroma_db"
MODEL_DIR   = BASE_DIR / "lora_model"
EVAL_DIR    = BASE_DIR / "evaluation"

for d in [BASE_DIR, CORPUS_DIR, DATA_DIR, CHROMA_DIR, MODEL_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

QA_PATH         = DATA_DIR / "qa_pairs.jsonl"
CHECKPOINT_PATH = BASE_DIR / "checkpoint.json"

# ── Checkpoint helper ───────────────────────────────────────
def save_checkpoint(stage: int, info: dict = {}):
    data = {"stage_completed": stage, **info}
    CHECKPOINT_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2))
    print(f"💾 Checkpoint salvo: etapa {stage} concluída")

def load_checkpoint() -> dict:
    if CHECKPOINT_PATH.exists():
        return json.loads(CHECKPOINT_PATH.read_text())
    return {"stage_completed": 0}

ckpt = load_checkpoint()
print(f"📌 Última etapa concluída: {ckpt.get('stage_completed', 0)}")
print(f"🖥️  GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Modelo: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"\n📁 BASE_DIR: {BASE_DIR}")


📌 Última etapa concluída: 0
🖥️  GPU disponível: True
   Modelo: Tesla T4
   VRAM: 15.6 GB

📁 BASE_DIR: /content/projeto_edificios_verdes


---
## 📋 ETAPA 1 — Planejamento e Escopo

### Recorte Temático
Abrangência total: eficiência hídrica, eficiência energética, certificações e tecnologias habilitadoras.

### Decisões Tecnológicas

| Componente | Escolha | Justificativa |
|---|---|---|
| Modelo Base | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` | 3B parâmetros cabe na T4 (16 GB VRAM); pré-quantizado 4-bit pelo Unsloth sem necessidade de auth Meta |
| Estratégia de ajuste | QLoRA (r=16, α=32) | Treinamento em 4-bit com bitsandbytes; adapta apenas ≈0.5% dos parâmetros; memória ≈6 GB |
| Biblioteca de treino | `unsloth` + `trl.SFTTrainer` | Unsloth oferece 2× mais velocidade que HuggingFace puro na T4 |
| Embedding | `intfloat/multilingual-e5-large` | Recomendado pelo enunciado; suporta português técnico; 560 MB |
| Banco vetorial | ChromaDB (persistência em disco) | Open-source, zero configuração, API simples, suporta metadados |
| Interface | Gradio | Funciona nativamente em Colab (ngrok automático); menos dependências que Streamlit |
| Métricas | ROUGE-L + BERTScore | ROUGE-L para similaridade léxica; BERTScore para similaridade semântica |

### Hiperparâmetros Previstos
- Learning rate: 2e-4 (cosine schedule)
- Epochs: 3
- Batch size: 2 por dispositivo + gradient_accumulation=8 (batch efetivo = 16)
- max_seq_length: 1024 tokens
- LoRA rank r=16, alpha=32 → lambda_LoRA = alpha/r = 2
- Warmup: 10 steps


---
## 📚 ETAPA 2 — Construção do Corpus

10 documentos técnicos em 3 categorias:
- **Normas/Certificações**: LEED v4.1, AQUA-HQE, GBC Brasil EDGE, ABNT NBR 15575
- **Relatórios Técnicos**: PROCEL Edifica, IEA Net Zero, Sistemas FV (ABSOLAR/ANEEL)
- **Manuais de Tecnologia**: Reuso de Águas Cinzas, BEMS/Automação Predial


In [ ]:
import urllib.request, json

# ── Metadados do corpus ─────────────────────────────────────
CORPUS_META = {
    "doc_01_leed_eficiencia_hidrica.txt": {
        "fonte": "USGBC", "categoria": "norma_certificacao",
        "subcategoria": "agua", "ano": 2023, "vigencia": "vigente"
    },
    "doc_02_leed_energia_atmosfera.txt": {
        "fonte": "USGBC", "categoria": "norma_certificacao",
        "subcategoria": "energia", "ano": 2023, "vigencia": "vigente"
    },
    "doc_03_aqua_hqe_referencial.txt": {
        "fonte": "Fundacao Vanzolini/Cerway", "categoria": "norma_certificacao",
        "subcategoria": "energia_agua", "ano": 2022, "vigencia": "vigente"
    },
    "doc_04_gbc_brasil_edge.txt": {
        "fonte": "GBC Brasil / IFC", "categoria": "norma_certificacao",
        "subcategoria": "energia_agua", "ano": 2023, "vigencia": "vigente"
    },
    "doc_05_nbr15575_desempenho.txt": {
        "fonte": "ABNT", "categoria": "norma_certificacao",
        "subcategoria": "energia", "ano": 2021, "vigencia": "vigente"
    },
    "doc_06_sistemas_fotovoltaicos.txt": {
        "fonte": "ABSOLAR/ANEEL", "categoria": "relatorio_tecnico",
        "subcategoria": "energia", "ano": 2024, "vigencia": "vigente"
    },
    "doc_07_aguas_cinzas_reuso.txt": {
        "fonte": "ABNT/ANA", "categoria": "manual_tecnologia",
        "subcategoria": "agua", "ano": 2023, "vigencia": "vigente"
    },
    "doc_08_bems_automacao.txt": {
        "fonte": "ASHRAE/ABNT/ISO", "categoria": "manual_tecnologia",
        "subcategoria": "energia", "ano": 2023, "vigencia": "vigente"
    },
    "doc_09_procel_edifica.txt": {
        "fonte": "INMETRO/PROCEL", "categoria": "relatorio_tecnico",
        "subcategoria": "energia", "ano": 2023, "vigencia": "vigente"
    },
    "doc_10_net_zero_iea.txt": {
        "fonte": "IEA/GBPN", "categoria": "relatorio_tecnico",
        "subcategoria": "energia_agua", "ano": 2023, "vigencia": "vigente"
    },
}

# ── Verificar arquivos do corpus ─────────────────────────────
docs_ok = []
for fname, meta in CORPUS_META.items():
    fpath = CORPUS_DIR / fname
    if fpath.exists():
        size = fpath.stat().st_size
        docs_ok.append(fname)
        print(f"✅ {fname[:40]:40s} {size:>6d} bytes | {meta['subcategoria']}")
    else:
        print(f"❌ FALTANDO: {fname}")
        print("   → Copie os arquivos do corpus para:", CORPUS_DIR)

print(f"\n📊 Corpus: {len(docs_ok)}/10 documentos encontrados")

# Categorias
cats = {}
for f, m in CORPUS_META.items():
    cats.setdefault(m["categoria"], []).append(f)
print("\nDistribuição por categoria:")
for cat, fs in cats.items():
    print(f"  {cat}: {len(fs)} documentos")


✅ doc_01_leed_eficiencia_hidrica.txt         6762 bytes | agua
✅ doc_02_leed_energia_atmosfera.txt          6444 bytes | energia
✅ doc_03_aqua_hqe_referencial.txt            7260 bytes | energia_agua
✅ doc_04_gbc_brasil_edge.txt                 3824 bytes | energia_agua
✅ doc_05_nbr15575_desempenho.txt             5131 bytes | energia
✅ doc_06_sistemas_fotovoltaicos.txt          6738 bytes | energia
✅ doc_07_aguas_cinzas_reuso.txt              7003 bytes | agua
✅ doc_08_bems_automacao.txt                  5680 bytes | energia
✅ doc_09_procel_edifica.txt                  5674 bytes | energia
✅ doc_10_net_zero_iea.txt                    6755 bytes | energia_agua

📊 Corpus: 10/10 documentos encontrados

Distribuição por categoria:
  norma_certificacao: 5 documentos
  relatorio_tecnico: 3 documentos
  manual_tecnologia: 2 documentos


In [ ]:
# ── Verificar/baixar corpus se não existir ────────────────────
# (Em ambiente Colab: faça upload manual dos arquivos para CORPUS_DIR)
# Aqui verificamos e mostramos instruções

missing = [f for f in CORPUS_META if not (CORPUS_DIR / f).exists()]
if missing:
    print("📋 INSTRUÇÃO: Carregue os arquivos de corpus para o Colab:")
    print(f"   Destino: {CORPUS_DIR.absolute()}")
    print("\n   Opção rápida — monte o Google Drive e copie:")
    print("   from google.colab import drive")
    print("   drive.mount('/content/drive')")
    print(f"   !cp -r '/content/drive/MyDrive/corpus_edificios/*' {CORPUS_DIR}/")
else:
    print("✅ Todos os documentos do corpus encontrados!")
    save_checkpoint(2, {"docs": len(CORPUS_META)})


✅ Todos os documentos do corpus encontrados!
💾 Checkpoint salvo: etapa 2 concluída


---
## 🧹 ETAPA 3 — Limpeza, Normalização e Pares Q&A

### Pipeline de Pré-processamento
1. Extração de texto de cada documento
2. Remoção de ruídos (cabeçalhos repetidos, números de página, encoding)
3. Normalização de espaços e caracteres especiais
4. Preservação de tabelas e requisitos numerados
5. Geração/verificação dos pares Q&A no formato chat (SFTTrainer)


In [ ]:
import re, unicodedata

def clean_text(text: str) -> str:
    """Pipeline de limpeza para documentos técnicos em português."""
    # 1. Normalizar encoding Unicode
    text = unicodedata.normalize("NFC", text)

    # 2. Remover cabeçalhos/rodapés repetidos (linhas com apenas números ou símbolos)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # 3. Remover URLs isoladas em linha
    text = re.sub(r"^\s*https?://\S+\s*$", "", text, flags=re.MULTILINE)

    # 4. Normalizar espaços múltiplos e tabs
    text = re.sub(r"[ \t]+", " ", text)

    # 5. Normalizar quebras de linha excessivas (>2)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 6. Remover linhas de apenas pontuação/traços (separadores)
    text = re.sub(r"^[\-=_*~]{5,}\s*$", "", text, flags=re.MULTILINE)

    # 7. Remover espaços no início/fim de linhas
    lines = [l.rstrip() for l in text.split("\n")]
    text = "\n".join(lines)

    return text.strip()

def extract_sections(text: str) -> list:
    """Extrai seções numeradas para preservar estrutura técnica."""
    # Identifica títulos de seção (1. TÍTULO, 2.1 SUBTÍTULO, etc.)
    pattern = re.compile(r"(\n\d+\.\d*\.?\s+[A-ZÁÉÍÓÚÀÂÊÔÃÕÇ][A-ZÁÉÍÓÚÀÂÊÔÃÕÇ\s]+)")
    sections = pattern.split(text)
    return sections

# Processar corpus
processed_docs = {}
for fname, meta in CORPUS_META.items():
    fpath = CORPUS_DIR / fname
    if not fpath.exists():
        continue
    raw = fpath.read_text(encoding="utf-8", errors="replace")
    clean = clean_text(raw)
    processed_docs[fname] = {"text": clean, "meta": meta, "chars": len(clean)}
    print(f"✅ {fname[:35]:35s} {len(raw):>6d} → {len(clean):>6d} chars")

print(f"\n📊 {len(processed_docs)} documentos processados")
print(f"   Total de caracteres: {sum(d['chars'] for d in processed_docs.values()):,}")


✅ doc_01_leed_eficiencia_hidrica.txt    6473 →   6357 chars
✅ doc_02_leed_energia_atmosfera.txt     6188 →   6053 chars
✅ doc_03_aqua_hqe_referencial.txt       6880 →   6696 chars
✅ doc_04_gbc_brasil_edge.txt            3659 →   3543 chars
✅ doc_05_nbr15575_desempenho.txt        4882 →   4497 chars
✅ doc_06_sistemas_fotovoltaicos.txt     6405 →   6224 chars
✅ doc_07_aguas_cinzas_reuso.txt         6620 →   6482 chars
✅ doc_08_bems_automacao.txt             5508 →   5364 chars
✅ doc_09_procel_edifica.txt             5361 →   5245 chars
✅ doc_10_net_zero_iea.txt               6451 →   6335 chars

📊 10 documentos processados
   Total de caracteres: 56,796


In [ ]:
# ── Carregar e validar pares Q&A ──────────────────────────────
qa_pairs = []
if QA_PATH.exists():
    with open(QA_PATH, encoding="utf-8") as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line.strip())
                # Validar estrutura esperada pelo SFTTrainer
                assert "messages" in obj
                assert len(obj["messages"]) == 3
                assert obj["messages"][0]["role"] == "system"
                assert obj["messages"][1]["role"] == "user"
                assert obj["messages"][2]["role"] == "assistant"
                qa_pairs.append(obj)
            except Exception as e:
                print(f"  ⚠️  Linha {i+1} inválida: {e}")

    print(f"✅ {len(qa_pairs)} pares Q&A válidos carregados")
else:
    print("❌ Arquivo qa_pairs.jsonl não encontrado!")
    print(f"   Esperado em: {QA_PATH}")

# Estatísticas dos pares
if qa_pairs:
    q_lens = [len(p["messages"][1]["content"]) for p in qa_pairs]
    a_lens = [len(p["messages"][2]["content"]) for p in qa_pairs]
    print(f"\n📊 Estatísticas dos pares Q&A:")
    print(f"   Tamanho médio da pergunta: {sum(q_lens)/len(q_lens):.0f} chars")
    print(f"   Tamanho médio da resposta: {sum(a_lens)/len(a_lens):.0f} chars")
    print(f"   Resposta mais curta: {min(a_lens)} | mais longa: {max(a_lens)} chars")


✅ 492 pares Q&A válidos carregados

📊 Estatísticas dos pares Q&A:
   Tamanho médio da pergunta: 87 chars
   Tamanho médio da resposta: 818 chars
   Resposta mais curta: 217 | mais longa: 1484 chars


In [ ]:
# ── Formatar pares para SFTTrainer ─────────────────────────────
from datasets import Dataset

def format_for_training(example):
    """Formata o par Q&A no template de chat Llama 3."""
    msgs = example["messages"]
    # Formato Llama 3 Instruct
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{msgs[0]['content']}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{msgs[1]['content']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{msgs[2]['content']}<|eot_id|>"
        "<|end_of_text|>"
    )
    return {"text": text}

# Criar dataset HuggingFace
raw_ds = Dataset.from_list(qa_pairs)
train_dataset = raw_ds.map(format_for_training, remove_columns=raw_ds.column_names)

# Divisão treino/validação (90/10)
split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"✅ Dataset preparado:")
print(f"   Treino: {len(train_ds)} exemplos")
print(f"   Validação: {len(val_ds)} exemplos")
print(f"\nExemplo de entrada formatada (truncado):")
print(train_ds[0]["text"][:400] + "...")

save_checkpoint(3, {"qa_pairs": len(qa_pairs), "train_size": len(train_ds)})


Map:   0%|          | 0/492 [00:00<?, ? examples/s]

✅ Dataset preparado:
   Treino: 442 exemplos
   Validação: 50 exemplos

Exemplo de entrada formatada (truncado):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Você é um assistente especialista em edifícios verdes, sustentáveis e Net Zero, com profundo conhecimento em certificações LEED, AQUA-HQE, GBC Brasil EDGE, normas ABNT, eficiência energética, energia solar fotovoltaica, reaproveitamento de águas cinzas e pluviais, BEMS e edificações Net Zero. Responda de forma técnica, precisa e em portu...
💾 Checkpoint salvo: etapa 3 concluída


---
## 🔢 ETAPA 4 — Chunking, Embeddings e Banco Vetorial

### Parâmetros
- Tamanho do chunk: 512 tokens (≈ 400 palavras)
- Sobreposição: 25% entre chunks adjacentes (≈ 128 tokens)
- Respeito à estrutura semântica: cortes apenas entre parágrafos/seções
- Modelo de embedding: `intfloat/multilingual-e5-large`
- Banco vetorial: ChromaDB com persistência em disco


In [ ]:
import textwrap

def semantic_chunker(text: str, max_chars: int = 2000, overlap_chars: int = 400) -> list:
    """
    Segmenta texto respeitando parágrafos e seções numeradas.
    max_chars ≈ 512 tokens (1 token ≈ 4 chars em pt)
    overlap_chars ≈ 25% de sobreposição
    """
    # Dividir em parágrafos
    paragraphs = re.split(r"\n{2,}", text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    chunks = []
    current_chunk = []
    current_len = 0

    for para in paragraphs:
        para_len = len(para)

        # Parágrafo maior que max_chars → dividir em frases
        if para_len > max_chars:
            sentences = re.split(r"(?<=[.!?;])\s+", para)
            for sent in sentences:
                if current_len + len(sent) > max_chars and current_chunk:
                    chunks.append(" ".join(current_chunk))
                    # Sobreposição: manter últimas frases
                    overlap_text = " ".join(current_chunk)[-overlap_chars:]
                    current_chunk = [overlap_text] if overlap_text else []
                    current_len = len(overlap_text)
                current_chunk.append(sent)
                current_len += len(sent) + 1
        else:
            if current_len + para_len > max_chars and current_chunk:
                chunks.append("\n\n".join(current_chunk))
                # Sobreposição: último parágrafo completo
                if current_chunk:
                    overlap = current_chunk[-1] if len(current_chunk[-1]) <= overlap_chars else current_chunk[-1][-overlap_chars:]
                    current_chunk = [overlap]
                    current_len = len(overlap)
                else:
                    current_chunk = []
                    current_len = 0
            current_chunk.append(para)
            current_len += para_len + 2

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return [c for c in chunks if len(c.strip()) > 50]  # Descartar chunks muito pequenos

# Aplicar chunking a todos os documentos
all_chunks = []
for fname, doc in processed_docs.items():
    chunks = semantic_chunker(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "id": f"{fname}_{i:03d}",
            "text": chunk,
            "source": fname,
            **doc["meta"]
        })

print(f"✅ Total de chunks gerados: {len(all_chunks)}")
print(f"\nDistribuição por categoria:")
from collections import Counter
cats = Counter(c["categoria"] for c in all_chunks)
for cat, n in cats.items():
    print(f"  {cat}: {n} chunks")

lens = [len(c["text"]) for c in all_chunks]
print(f"\nTamanho médio: {sum(lens)/len(lens):.0f} chars")
print(f"Tamanho mínimo: {min(lens)} | máximo: {max(lens)} chars")


✅ Total de chunks gerados: 36

Distribuição por categoria:
  norma_certificacao: 17 chunks
  relatorio_tecnico: 11 chunks
  manual_tecnologia: 8 chunks

Tamanho médio: 1750 chars
Tamanho mínimo: 564 | máximo: 1996 chars


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# ── Carregar modelo de embedding ────────────────────────────
print("⏳ Carregando modelo de embedding (multilingual-e5-large, ~560 MB)...")
print("   Isso pode levar 1-3 minutos na primeira vez...")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")

print(f"✅ Modelo carregado! Dimensão dos embeddings: {embedding_model.get_sentence_embedding_dimension()}")

# ── Gerar embeddings ─────────────────────────────────────────
# Prefixo "query: " e "passage: " são recomendados para multilingual-e5
texts = [f"passage: {c['text']}" for c in all_chunks]

print(f"\n⏳ Gerando embeddings para {len(texts)} chunks...")
batch_size = 32

embeddings = embedding_model.encode(
    texts,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 norm para cosine similarity
)

print(f"✅ Embeddings gerados: shape {embeddings.shape}")
print(f"   dtype: {embeddings.dtype}")


⏳ Carregando modelo de embedding (multilingual-e5-large, ~560 MB)...
   Isso pode levar 1-3 minutos na primeira vez...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Modelo carregado! Dimensão dos embeddings: 1024

⏳ Gerando embeddings para 36 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Embeddings gerados: shape (36, 1024)
   dtype: float32


In [ ]:
import chromadb

# ── Indexar no ChromaDB ─────────────────────────────────────
print("⏳ Criando coleção ChromaDB...")

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Deletar coleção existente se houver (para re-execução limpa)
try:
    chroma_client.delete_collection("edificios_verdes")
    print("   (Coleção anterior removida)")
except:
    pass

collection = chroma_client.create_collection(
    name="edificios_verdes",
    metadata={"hnsw:space": "cosine"}
)

# Inserir em batches
BATCH = 100
for i in range(0, len(all_chunks), BATCH):
    batch_chunks = all_chunks[i:i+BATCH]
    batch_embs   = embeddings[i:i+BATCH]

    collection.add(
        ids=[c["id"] for c in batch_chunks],
        documents=[c["text"] for c in batch_chunks],
        embeddings=batch_embs.tolist(),
        metadatas=[{
            "source": c["source"],
            "categoria": c["categoria"],
            "subcategoria": c["subcategoria"],
            "ano": c["ano"]
        } for c in batch_chunks]
    )
    print(f"   Indexados {min(i+BATCH, len(all_chunks))}/{len(all_chunks)} chunks...")

print(f"\n✅ ChromaDB indexado em: {CHROMA_DIR}")
print(f"   Total de vetores: {collection.count()}")

# ── Teste de recuperação ─────────────────────────────────────
print("\n🔍 Teste de recuperação:")
query = "query: Qual é o percentual mínimo de redução de água exigido pelo LEED?"
q_emb = embedding_model.encode([query], normalize_embeddings=True)[0]

results = collection.query(
    query_embeddings=[q_emb.tolist()],
    n_results=3
)

for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"\n  Resultado {i+1} (similaridade: {1-dist:.3f}):")
    print(f"  {doc[:200]}...")

save_checkpoint(4, {"chunks": len(all_chunks), "vectors": collection.count()})


⏳ Criando coleção ChromaDB...
   Indexados 36/36 chunks...

✅ ChromaDB indexado em: /content/projeto_edificios_verdes/chroma_db
   Total de vetores: 36

🔍 Teste de recuperação:

  Resultado 1 (similaridade: 0.870):
  FONTE: USGBC - U.S. Green Building Council
CATEGORIA: norma_certificacao
SUBCATEGORIA: agua
ANO: 2023
VIGENCIA: vigente
TITULO: LEED v4.1 BD+C - Eficiência Hídrica: Pré-requisitos e Créditos

LEED v4....

  Resultado 2 (similaridade: 0.858):
  Fórmula de cálculo da linha de base:
Volume diário (L) = Nº de ocupantes × Usos por fixture/dia × Volume por uso (L)
Exemplo: 100 ocupantes, 3 usos de vaso/dia × 6,0 L = 1.800 L/dia de baseline para v...

  Resultado 3 (similaridade: 0.854):
  Dados de medição devem ser registrados em intervalos de 15 minutos a 1 hora, com transmissão para sistema de BEMS ou BAS (Building Automation System).

7. INTEGRAÇÃO COM NET ZERO WATER
Para atingir ba...
💾 Checkpoint salvo: etapa 4 concluída


In [ ]:
# ── Relatório da Etapa 4 ────────────────────────────────────
import json
from collections import defaultdict

report = {
    "total_chunks": len(all_chunks),
    "total_vectores_chromadb": collection.count(),
    "distribuicao_categoria": dict(Counter(c["categoria"] for c in all_chunks)),
    "distribuicao_subcategoria": dict(Counter(c["subcategoria"] for c in all_chunks)),
    "tamanho_medio_chars": int(sum(len(c["text"]) for c in all_chunks) / len(all_chunks)),
    "tamanho_medio_tokens_est": int(sum(len(c["text"]) for c in all_chunks) / len(all_chunks) / 4),
    "modelo_embedding": "intfloat/multilingual-e5-large",
    "dimensao_embedding": embedding_model.get_sentence_embedding_dimension(),
    "banco_vetorial": "ChromaDB (persistência local)",
}

print("=" * 60)
print("RELATÓRIO — ETAPA 4: CHUNKING E EMBEDDINGS")
print("=" * 60)
for k, v in report.items():
    print(f"  {k}: {v}")

(DATA_DIR / "relatorio_chunking.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2)
)
print("\n✅ Relatório salvo em data/relatorio_chunking.json")


RELATÓRIO — ETAPA 4: CHUNKING E EMBEDDINGS
  total_chunks: 36
  total_vectores_chromadb: 36
  distribuicao_categoria: {'norma_certificacao': 17, 'relatorio_tecnico': 11, 'manual_tecnologia': 8}
  distribuicao_subcategoria: {'agua': 8, 'energia': 18, 'energia_agua': 10}
  tamanho_medio_chars: 1750
  tamanho_medio_tokens_est: 437
  modelo_embedding: intfloat/multilingual-e5-large
  dimensao_embedding: 1024
  banco_vetorial: ChromaDB (persistência local)

✅ Relatório salvo em data/relatorio_chunking.json


---
## 🤖 ETAPA 5 — Fine-Tuning com QLoRA

### Configuração
- **Modelo base**: `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`
- **Estratégia**: QLoRA — LoRA sobre pesos quantizados em 4-bit
- **Justificativa de hiperparâmetros**:
  - `r=16`: rank moderado, bom equilíbrio capacidade/memória
  - `alpha=32`: λ = alpha/r = 2 → escalonamento padrão
  - `lr=2e-4`: taxa recomendada pelo Unsloth para QLoRA
  - `epochs=3`: suficiente para 163 pares sem overfitting
  - `batch_efetivo=16` (2 × grad_accum 8): simula batch maior na T4

⏱️ **Tempo estimado na T4**: 25-40 minutos para 3 épocas


In [ ]:
from unsloth import FastLanguageModel

# ── Carregar modelo base com Unsloth ───────────────────────
print("⏳ Carregando Llama 3.2 3B Instruct (4-bit quantizado)...")
print("   Download na primeira execução: ~2 GB")
print("   Tempo estimado: 2-5 minutos...")

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # Auto-detect (bf16 se suportado, fp16 caso contrário)
    load_in_4bit=True,
)

print("✅ Modelo base carregado!")
print(f"   Parâmetros totais: {model.num_parameters():,}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Carregando Llama 3.2 3B Instruct (4-bit quantizado)...
   Download na primeira execução: ~2 GB
   Tempo estimado: 2-5 minutos...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Modelo base carregado!
   Parâmetros totais: 3,212,749,824


In [ ]:
# ── Aplicar LoRA (PEFT) ─────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,           # 0 é ótimo com Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # economiza 30% de VRAM
    random_state=42,
    use_rslora=False,         # LoRA padrão
    loftq_config=None,
)

# Resumo dos parâmetros treináveis
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ QLoRA aplicado!")
print(f"   Parâmetros treináveis: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Parâmetros congelados: {total-trainable:,}")


Unsloth 2026.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ QLoRA aplicado!
   Parâmetros treináveis: 24,313,856 (1.33%)
   Parâmetros congelados: 1,803,463,680


In [ ]:
from trl import SFTTrainer, SFTConfig  # SFTConfig resolve o PicklingError
import torch

# ── SFTConfig centraliza TrainingArguments + campos do SFTTrainer ──────────
sft_config = SFTConfig(
    output_dir=str(BASE_DIR / "outputs"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,     # batch efetivo = 16
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",                 # economiza memória
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none",                   # sem WandB
    dataloader_num_workers=0,
    # Campos específicos do SFTConfig (antes ficavam no SFTTrainer)
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=1,
    packing=False,                      # sem packing para datasets pequenos
)

# SFTTrainer recebe apenas model, dados e args — sem campos duplicados
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
)

print("✅ SFTTrainer configurado!")
steps_per_epoch = len(train_ds) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
print(f"   Steps por época: {steps_per_epoch}")
print(f"   Total de steps: ~{3 * steps_per_epoch}")


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/442 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/50 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ SFTTrainer configurado!
   Steps por época: 27
   Total de steps: ~81


In [ ]:
# ── Salvar respostas do modelo BASE (para comparação pós-treino) ─
PERGUNTAS_AVALIACAO = [
    "Qual é o percentual mínimo de redução de água interior exigido pelo pré-requisito WEp1 do LEED v4.1?",
    "Como calcular a potência de pico de um sistema fotovoltaico para uma edificação?",
    "O que é a certificação AQUA-HQE e quais são seus perfis de desempenho?",
    "Quais são os parâmetros de qualidade exigidos pela NBR 16783 para reuso de água em descarga de vasos?",
    "Qual é o EUI alvo para um escritório Net Zero Energy?",
    "Como funciona o BEMS e quais protocolos de comunicação são utilizados?",
    "O que é QLoRA e como difere do LoRA convencional no fine-tuning de LLMs?",
    "Quais são as zonas bioclimáticas brasileiras e como influenciam os requisitos da NBR 15575?",
    "O que é o crédito EA2 do LEED e como é calculada a pontuação?",
    "Quais são as etapas de tratamento de águas cinzas para reuso em edificações?"
]

SYSTEM_PROMPT = (
    "Você é um assistente especialista em edifícios verdes, sustentáveis e Net Zero, "
    "com profundo conhecimento em certificações LEED, AQUA-HQE, GBC Brasil EDGE, "
    "normas ABNT, eficiência energética, energia solar fotovoltaica, "
    "reaproveitamento de águas cinzas e pluviais, BEMS e edificações Net Zero. "
    "Responda de forma técnica, precisa e em português."
)

def gerar_resposta(mdl, tok, pergunta, max_tokens=512):
    """Gera uma resposta usando o modelo atual."""
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pergunta}
    ]
    inputs = tok.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = mdl.generate(
            input_ids=inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tok.eos_token_id
        )

    new_tokens = outputs[0][inputs.shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

# Coletar respostas do modelo BASE
print("⏳ Coletando respostas do MODELO BASE (antes do fine-tuning)...")
FastLanguageModel.for_inference(model)

respostas_base = {}
for i, pergunta in enumerate(PERGUNTAS_AVALIACAO):
    print(f"  [{i+1}/10] {pergunta[:60]}...")
    respostas_base[pergunta] = gerar_resposta(model, tokenizer, pergunta)

# Salvar
(EVAL_DIR / "respostas_base.json").write_text(
    json.dumps(respostas_base, ensure_ascii=False, indent=2)
)
print("\n✅ Respostas do modelo BASE salvas em evaluation/respostas_base.json")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


⏳ Coletando respostas do MODELO BASE (antes do fine-tuning)...
  [1/10] Qual é o percentual mínimo de redução de água interior exigi...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [2/10] Como calcular a potência de pico de um sistema fotovoltaico ...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [8/10] Quais são as zonas bioclimáticas brasileiras e como influenc...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [9/10] O que é o crédito EA2 do LEED e como é calculada a pontuação...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [10/10] Quais são as etapas de tratamento de águas cinzas para reuso...

✅ Respostas do modelo BASE salvas em evaluation/respostas_base.json


In [ ]:
# ── TREINAMENTO ─────────────────────────────────────────────
print("🚀 Iniciando fine-tuning QLoRA...")
print("   Tempo estimado: 25-45 minutos na T4")
print("   Acompanhe o loss a cada 10 steps\n")

import time
t0 = time.time()

# Re-ativar modo de treinamento
model.train()
trainer_stats = trainer.train()

elapsed = time.time() - t0
print(f"\n✅ Treinamento concluído em {elapsed/60:.1f} minutos!")
print(f"   Loss final (treino): {trainer_stats.training_loss:.4f}")


🚀 Iniciando fine-tuning QLoRA...
   Tempo estimado: 25-45 minutos na T4
   Acompanhe o loss a cada 10 steps



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 442 | Num Epochs = 3 | Total steps = 84
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.532505
20,1.617542
30,1.432068
40,1.335288
50,1.272367
60,1.208411
70,1.150784
80,1.123331


Unsloth: Restored added_tokens_decoder metadata in /content/projeto_edificios_verdes/outputs/checkpoint-28/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/projeto_edificios_verdes/outputs/checkpoint-56/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/projeto_edificios_verdes/outputs/checkpoint-84/tokenizer_config.json.



✅ Treinamento concluído em 17.4 minutos!
   Loss final (treino): 1.4433


In [ ]:
# ── Salvar adaptador LoRA ───────────────────────────────────
print("💾 Salvando adaptador LoRA...")
model.save_pretrained(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print(f"✅ Adaptador LoRA salvo em: {MODEL_DIR}")
print(f"   Arquivos:")
for f in sorted(MODEL_DIR.iterdir()):
    print(f"   {f.name:40s} {f.stat().st_size/1024:.1f} KB")

save_checkpoint(5, {
    "model_dir": str(MODEL_DIR),
    "loss_final": round(trainer_stats.training_loss, 4)
})


💾 Salvando adaptador LoRA...


Unsloth: Restored added_tokens_decoder metadata in /content/projeto_edificios_verdes/lora_model/tokenizer_config.json.


✅ Adaptador LoRA salvo em: /content/projeto_edificios_verdes/lora_model
   Arquivos:
   README.md                                5.1 KB
   adapter_config.json                      1.2 KB
   adapter_model.safetensors                95026.9 KB
   chat_template.jinja                      3.7 KB
   tokenizer.json                           16806.6 KB
   tokenizer_config.json                    49.5 KB
💾 Checkpoint salvo: etapa 5 concluída


---
## 📊 ETAPA 6 — Avaliação do Modelo Fine-Tunado

### Metodologia
1. Carregar o modelo fine-tunado
2. Gerar respostas para as 10 perguntas de avaliação
3. Comparar com respostas do modelo base (coletadas antes do treino)
4. Calcular ROUGE-L e BERTScore
5. Análise qualitativa das melhorias e limitações


In [ ]:
from unsloth import FastLanguageModel
import json

# Recarregar modelo fine-tunado
print("⏳ Carregando modelo fine-tunado...")
model_ft, tokenizer_ft = FastLanguageModel.from_pretrained(
    model_name=str(MODEL_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_ft)
print("✅ Modelo fine-tunado carregado!")

# Carregar respostas do modelo base
respostas_base = json.loads((EVAL_DIR / "respostas_base.json").read_text())

# Gerar respostas do modelo fine-tunado
print("\n⏳ Coletando respostas do MODELO FINE-TUNADO...")
respostas_ft = {}
for i, pergunta in enumerate(PERGUNTAS_AVALIACAO):
    print(f"  [{i+1}/10] {pergunta[:60]}...")
    respostas_ft[pergunta] = gerar_resposta(model_ft, tokenizer_ft, pergunta)

(EVAL_DIR / "respostas_ft.json").write_text(
    json.dumps(respostas_ft, ensure_ascii=False, indent=2)
)
print("\n✅ Respostas do modelo FT salvas em evaluation/respostas_ft.json")


⏳ Carregando modelo fine-tunado...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Modelo fine-tunado carregado!

⏳ Coletando respostas do MODELO FINE-TUNADO...
  [1/10] Qual é o percentual mínimo de redução de água interior exigi...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [8/10] Quais são as zonas bioclimáticas brasileiras e como influenc...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [9/10] O que é o crédito EA2 do LEED e como é calculada a pontuação...


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [10/10] Quais são as etapas de tratamento de águas cinzas para reuso...

✅ Respostas do modelo FT salvas em evaluation/respostas_ft.json


In [ ]:
# ── Respostas de referência (escritas pelo aluno) ─────────────
# INSTRUÇÃO: Complete com suas respostas de referência para cada pergunta
RESPOSTAS_REFERENCIA = {
    PERGUNTAS_AVALIACAO[0]: "O pré-requisito WEp1 do LEED v4.1 exige redução mínima de 20% no consumo de água interior em relação à linha de base estabelecida pelo Energy Policy Act (EPAct) de 1992. Para vasos sanitários, a baseline é de 6,0 L/descarga e o máximo permitido é 4,8 L. Para mictórios, baseline de 3,8 L e máximo de 1,9 L. Torneiras de lavatório: baseline 8,3 L/min e máximo 5,7 L/min em uso residencial, ou temporizada 8 seg para uso transiente.",
    PERGUNTAS_AVALIACAO[1]: "A potência de pico é calculada por: Ppico (kWp) = Consumo diário (kWh/dia) / (ISPG × PR), onde ISPG é a Irradiação Solar no Plano do Gerador (kWh/m²/dia) obtida no Atlas CRESESB e PR é o Performance Ratio (0,80-0,85), que representa as perdas do sistema (inversores, cabeamento, sujidade, temperatura). O número de módulos = Ppico × 1000 / Potência por módulo (Wp).",
    PERGUNTAS_AVALIACAO[2]: "O AQUA-HQE é o sistema de certificação verde brasileiro adaptado do HQE francês pela Fundação Vanzolini. Avalia 14 preocupações (QB1-QB14) organizadas em quatro famílias: Eco-Construção (QB1-QB4), Eco-Gestão (QB5-QB8), Conforto (QB9-QB12) e Saúde (QB13-QB14). Cada preocupação é avaliada em três perfis: BASE (mínimo regulatório), BOAS PRÁTICAS (acima do mínimo) e MELHORES PRÁTICAS (excelência do mercado). Os níveis de certificação vão de 'Bom' a 'Excepcional'.",
    PERGUNTAS_AVALIACAO[3]: "A NBR 16783:2019 define para Classe B (descarga de vasos sanitários): Coliformes termotolerantes ≤ 200 NMP/100 mL, turbidez ≤ 5 NTU, pH entre 6,0 e 9,0, cloro residual livre entre 0,5 e 1,5 mg/L e DBO₅ ≤ 30 mg/L. Para usos com contato humano (Classe A): coliformes termotolerantes ausentes em 100 mL, turbidez ≤ 5 NTU e DBO₅ ≤ 10 mg/L.",
    PERGUNTAS_AVALIACAO[4]: "O EUI (Energy Use Intensity) alvo para escritórios Net Zero Energy é de ≤ 50 kWh/m²/ano, com benchmarks avançados chegando a 30 kWh/m²/ano. Isso representa uma redução de 60-75% em relação ao EUI de referência ASHRAE 90.1-2016 para escritórios no Brasil (ZB3: ~120-150 kWh/m²/ano). Para atingir o balanço nulo, o sistema fotovoltaico deve gerar pelo menos tanto quanto o EUI anual multiplicado pela área do edifício.",
    PERGUNTAS_AVALIACAO[5]: "O BEMS (Building Energy Management System) é um sistema computadorizado de monitoramento e controle de sistemas prediais. Sua arquitetura é em três níveis: campo (sensores, atuadores, controladores DDC), automação (controladores de área e zona) e gestão (servidor BEMS, interface GUI, dashboards). Os principais protocolos são: BACnet (padrão ANSI/ASHRAE para HVAC), Modbus RTU/TCP (medidores de energia e inversores), KNX (iluminação e persianas) e MQTT (IoT e integração cloud).",
    PERGUNTAS_AVALIACAO[6]: "O crédito WE3 (Reuso de Água) do LEED v4.1 BD+C oferece até 2 pontos: 1 ponto para redução de 50% do consumo de água para irrigação paisagística ou eliminação total; 1 ponto adicional para reuso de pelo menos 50% da água interior (vasos e mictórios) por meio de água pluvial ou de reuso tratada. O cálculo compara o volume de água reutilizada versus o consumo de referência calculado pela ferramenta LEED Water Use Reduction Calculator, considerando a ocupação do edifício, dias de uso e fixtures instalados.",
    PERGUNTAS_AVALIACAO[7]: "O Brasil tem 8 zonas bioclimáticas (ZB1-ZB8 da NBR 15220-3): ZB1 (Sul frio: Caxias do Sul), ZB2 (Sul temperado: Florianópolis), ZB3 (Centro temperado: São Paulo, BH), ZB4 (Centro-Oeste intermediário: Brasília), ZB5 (Centro-Oeste quente: Campo Grande), ZB6 (Nordeste semi-árido), ZB7 (Litoral Norte/NE: Fortaleza) e ZB8 (Amazônia: Belém). A NBR 15575 define transmitâncias máximas de paredes e coberturas por zona; zonas quentes (ZB5-ZB8) exigem absortância ≤ 0,40 para paredes e coberturas.",
    PERGUNTAS_AVALIACAO[8]: "O crédito EA2 (Otimização do Desempenho Energético) do LEED v4.1 oferece até 18 pontos para novas construções, baseados na porcentagem de economia energética versus o edifício de referência ASHRAE 90.1-2016. A escala vai de 1 ponto (6% de economia) até 18 pontos (≥50% de economia). É calculado por simulação energética completa (Energy Modeling) comparando o EUI do projeto com o edifício de referência nas mesmas condições climáticas.",
    PERGUNTAS_AVALIACAO[9]: "As etapas de tratamento de águas cinzas incluem: (1) Gradeamento/peneiramento (1-3 mm) para remoção de sólidos grosseiros; (2) Flotação por ar dissolvido (DAF) ou decantação para remoção de gorduras e sólidos em suspensão (60-80% de eficiência); (3) Filtração em múltiplas camadas — areia grossa, areia fina e carvão ativado granular (GAC) para adsorção de surfactantes; (4) Desinfecção com cloro (1-3 mg/L de hipoclorito de sódio) ou UV (dose mínima 30 mJ/cm²). Para maior qualidade (Classe A), pode-se adicionar biorreator de membrana (MBR).",
}
print("✅ Respostas de referência carregadas!")

✅ Respostas de referência carregadas!


In [ ]:
from rouge_score import rouge_scorer as rs_module
import evaluate

# ── ROUGE-L ──────────────────────────────────────────────────
print("📐 Calculando ROUGE-L...")
scorer = rs_module.RougeScorer(["rougeL"], use_stemmer=False)

resultados = []
for pergunta in PERGUNTAS_AVALIACAO:
    ref   = RESPOSTAS_REFERENCIA[pergunta]
    base  = respostas_base.get(pergunta, "")
    ft    = respostas_ft.get(pergunta, "")

    score_base = scorer.score(ref, base)["rougeL"].fmeasure
    score_ft   = scorer.score(ref, ft)["rougeL"].fmeasure
    delta      = score_ft - score_base

    resultados.append({
        "pergunta": pergunta[:60] + "...",
        "rouge_l_base": round(score_base, 4),
        "rouge_l_ft":   round(score_ft, 4),
        "delta":         round(delta, 4)
    })

avg_base = sum(r["rouge_l_base"] for r in resultados) / len(resultados)
avg_ft   = sum(r["rouge_l_ft"]   for r in resultados) / len(resultados)

print(f"\n{'Pergunta':55s} {'Base':>8} {'FT':>8} {'Δ':>8}")
print("-" * 85)
for r in resultados:
    delta_str = f"+{r['delta']:.4f}" if r["delta"] >= 0 else f"{r['delta']:.4f}"
    print(f"{r['pergunta']:55s} {r['rouge_l_base']:>8.4f} {r['rouge_l_ft']:>8.4f} {delta_str:>8}")
print("-" * 85)
print(f"{'MÉDIA':55s} {avg_base:>8.4f} {avg_ft:>8.4f} {avg_ft-avg_base:>+8.4f}")
print(f"\n✅ Melhoria média ROUGE-L: {(avg_ft-avg_base)*100:+.1f} pontos percentuais")


📐 Calculando ROUGE-L...

Pergunta                                                    Base       FT        Δ
-------------------------------------------------------------------------------------
Qual é o percentual mínimo de redução de água interior exigi...   0.2500   0.2901  +0.0401
Como calcular a potência de pico de um sistema fotovoltaico ...   0.1013   0.1359  +0.0347
O que é a certificação AQUA-HQE e quais são seus perfis de d...   0.1190   0.1390  +0.0200
Quais são os parâmetros de qualidade exigidos pela NBR 16783...   0.1484   0.1489  +0.0005
Qual é o EUI alvo para um escritório Net Zero Energy?...   0.2491   0.3288  +0.0796
Como funciona o BEMS e quais protocolos de comunicação são u...   0.1569   0.2119  +0.0550
O que é QLoRA e como difere do LoRA convencional no fine-tun...   0.1749   0.2381  +0.0632
Quais são as zonas bioclimáticas brasileiras e como influenc...   0.0928   0.1303  +0.0375
O que é o crédito EA2 do LEED e como é calculada a pontuação...   0.1436   0.1576  +0

In [ ]:
# ── BERTScore ────────────────────────────────────────────────
print("\n📐 Calculando BERTScore (pode levar 2-3 minutos)...")
bertscore = evaluate.load("bertscore")

refs = [RESPOSTAS_REFERENCIA[p] for p in PERGUNTAS_AVALIACAO]
preds_base = [respostas_base.get(p, "") for p in PERGUNTAS_AVALIACAO]
preds_ft   = [respostas_ft.get(p, "") for p in PERGUNTAS_AVALIACAO]

bs_base = bertscore.compute(predictions=preds_base, references=refs, lang="pt")
bs_ft   = bertscore.compute(predictions=preds_ft,   references=refs, lang="pt")

avg_bs_base = sum(bs_base["f1"]) / len(bs_base["f1"])
avg_bs_ft   = sum(bs_ft["f1"]) / len(bs_ft["f1"])

print(f"\n  BERTScore F1 — Modelo BASE: {avg_bs_base:.4f}")
print(f"  BERTScore F1 — Modelo FT:   {avg_bs_ft:.4f}")
print(f"  Melhoria BERTScore: {(avg_bs_ft - avg_bs_base)*100:+.2f} pontos percentuais")

# Salvar resultados completos
eval_report = {
    "rouge_l_base": avg_base,
    "rouge_l_ft": avg_ft,
    "rouge_l_delta": avg_ft - avg_base,
    "bertscore_f1_base": avg_bs_base,
    "bertscore_f1_ft": avg_bs_ft,
    "bertscore_delta": avg_bs_ft - avg_bs_base,
    "detalhes": resultados,
    "respostas_base": respostas_base,
    "respostas_ft": respostas_ft,
    "referencias": RESPOSTAS_REFERENCIA,
}

(EVAL_DIR / "relatorio_avaliacao.json").write_text(
    json.dumps(eval_report, ensure_ascii=False, indent=2)
)
print("\n✅ Relatório de avaliação salvo em evaluation/relatorio_avaliacao.json")
save_checkpoint(6, {
    "rouge_l_base": round(avg_base, 4), "rouge_l_ft": round(avg_ft, 4),
    "bertscore_base": round(avg_bs_base, 4), "bertscore_ft": round(avg_bs_ft, 4)
})



📐 Calculando BERTScore (pode levar 2-3 minutos)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  BERTScore F1 — Modelo BASE: 0.6782
  BERTScore F1 — Modelo FT:   0.7075
  Melhoria BERTScore: +2.93 pontos percentuais

✅ Relatório de avaliação salvo em evaluation/relatorio_avaliacao.json
💾 Checkpoint salvo: etapa 6 concluída


---
## 💬 ETAPA 7 — Chatbot com Interface Gradio

### Funcionalidades
- Histórico de conversa dentro da sessão
- Retrieval-Augmented Generation (RAG) opcional com ChromaDB
- Modelo fine-tunado + conhecimento do corpus
- Interface pública via Gradio share link


In [ ]:
import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
import torch, json

# ── Carregar modelo fine-tunado ─────────────────────────────
print("⏳ Carregando modelo fine-tunado para o chatbot...")
model_chat, tokenizer_chat = FastLanguageModel.from_pretrained(
    model_name=str(MODEL_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_chat)
print("✅ Modelo carregado!")

# ── Carregar ChromaDB para RAG ──────────────────────────────
print("⏳ Carregando ChromaDB e modelo de embedding...")
chroma_client_chat = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection_chat    = chroma_client_chat.get_collection("edificios_verdes")
emb_model_chat     = SentenceTransformer("intfloat/multilingual-e5-large")
print(f"✅ ChromaDB: {collection_chat.count()} vetores carregados")

SYSTEM_PROMPT = (
    "Você é um assistente especialista em edifícios verdes, sustentáveis e Net Zero. "
    "Tem profundo conhecimento em certificações LEED, AQUA-HQE, GBC Brasil EDGE, "
    "normas ABNT, eficiência energética, energia solar fotovoltaica, "
    "reaproveitamento de águas cinzas e pluviais, BEMS e edificações Net Zero. "
    "Responda de forma técnica, precisa e em português. "
    "Quando possível, cite normas, valores numéricos e referências técnicas específicas."
)

def retrieve_context(query: str, n: int = 3) -> str:
    """Recupera contexto relevante do ChromaDB via RAG."""
    q_emb = emb_model_chat.encode(
        [f"query: {query}"], normalize_embeddings=True
    )[0]
    results = collection_chat.query(
        query_embeddings=[q_emb.tolist()], n_results=n
    )
    if results["documents"][0]:
        context = "\n\n---\n\n".join(results["documents"][0])
        return f"[Contexto do corpus técnico]\n{context}\n[Fim do contexto]"
    return ""

def chat_fn(message: str, history: list, use_rag: bool = True) -> str:
    """Função principal do chatbot com RAG opcional."""
    # Montar mensagens de histórico
    system_content = SYSTEM_PROMPT
    if use_rag:
        ctx = retrieve_context(message)
        if ctx:
            system_content = SYSTEM_PROMPT + "\n\n" + ctx

    messages = [{"role": "system", "content": system_content}]
    for user_msg, assistant_msg in history[-4:]:  # Manter últimas 4 trocas
        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})

    inputs = tokenizer_chat.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model_chat.generate(
            input_ids=inputs,
            max_new_tokens=600,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer_chat.eos_token_id,
        )

    new_tokens = outputs[0][inputs.shape[1]:]
    response = tokenizer_chat.decode(new_tokens, skip_special_tokens=True).strip()
    return response

# ── Interface Gradio ─────────────────────────────────────────
with gr.Blocks(title="🏗️ Assistente de Edifícios Verdes", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏗️ Assistente para Edifícios Verdes e Net Zero
    *Especialista em LEED · AQUA-HQE · GBC Brasil EDGE · Energia Solar · Reuso de Água · NZEBs*
    """)

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(height=500, label="Conversa")
            msg_box = gr.Textbox(
                placeholder="Digite sua pergunta técnica...",
                label="Pergunta",
                lines=2
            )
            with gr.Row():
                send_btn  = gr.Button("Enviar ↗", variant="primary")
                clear_btn = gr.Button("Limpar 🗑️")

        with gr.Column(scale=1):
            use_rag = gr.Checkbox(value=True, label="🔍 Usar RAG (corpus)")
            gr.Markdown("### Exemplos de perguntas:")
            examples = gr.Examples(
                examples=[
                    ["Qual é o requisito mínimo de redução de água do LEED v4.1?"],
                    ["Como dimensionar um sistema fotovoltaico para 500 m²?"],
                    ["Quais parâmetros de qualidade a NBR 16783 exige para reuso em vasos?"],
                    ["O que é QLoRA e como funciona?"],
                    ["Quais são as 14 preocupações do AQUA-HQE?"],
                    ["Como calcular a potência de pico de um sistema FV?"],
                    ["O que é BEMS e quais protocolos usa?"],
                    ["Qual EUI alvo para um edifício Net Zero?"],
                ],
                inputs=msg_box
            )

    def respond(message, chat_history, rag):
        if not message.strip():
            return chat_history, ""
        bot_response = chat_fn(message, chat_history, use_rag=rag)
        chat_history.append((message, bot_response))
        return chat_history, ""

    send_btn.click(respond, [msg_box, chatbot, use_rag], [chatbot, msg_box])
    msg_box.submit(respond, [msg_box, chatbot, use_rag], [chatbot, msg_box])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_box])

print("\n🚀 Iniciando chatbot...")
demo.launch(share=True, debug=False)
# O link público aparece acima — válido por 72 horas

save_checkpoint(7, {"chatbot": "deployed"})


⏳ Carregando modelo fine-tunado para o chatbot...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Modelo carregado!
⏳ Carregando ChromaDB e modelo de embedding...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ChromaDB: 36 vetores carregados

🚀 Iniciando chatbot...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4869a893a630b4c1c5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


💾 Checkpoint salvo: etapa 7 concluída


---
## 📝 ETAPA 8 — Relatório Crítico

*Complete as células abaixo com suas análises após executar o projeto completo.*


### 8.1 Dificuldades na Coleta e Preparação do Corpus

**Coleta de dados:**
- A principal dificuldade foi obter documentos técnicos em português com cobertura abrangente e livre de direitos autorais. Normas ABNT (NBR 15575, NBR 16783) são pagas e acessíveis apenas por assinatura, obrigando à criação de documentos sintéticos baseados no conteúdo técnico publicado — o que limita a fidelidade normativa mas viabiliza a execução do projeto sem infração de direitos.
- Documentos do PROCEL (RTQ-C e RTQ-R) são públicos mas em formato PDF com tabelas complexas, exigindo extração manual e normalização cuidadosa para eliminar ruídos de conversão.
- A cobertura temática equilibrada entre energia e água exigiu curadoria ativa: sem intervenção, documentos de energia (LEED EA, fotovoltaicos, BEMS) naturalmente superariam os de água em volume, enviesando o fine-tuning.

**Pré-processamento e chunking:**
- O corpus gerou apenas 36 chunks (distribuídos em 17 de normas/certificações, 11 de relatórios técnicos e 8 de manuais de tecnologia), com tamanho médio de 1.750 caracteres (~437 tokens). Embora dentro da faixa recomendada (512–1024 tokens), o número reduzido de chunks limitou a cobertura do banco vetorial — perguntas muito específicas por vezes recuperam contexto parcialmente relacionado.
- Preservar tabelas de parâmetros técnicos durante o chunking foi desafiador: tabelas de transmitâncias por zona bioclimática e pontuações do LEED foram fragmentadas nas primeiras versões, comprometendo o contexto semântico. A solução adotada foi respeitar quebras de parágrafo como fronteiras de chunk.
- A tokenização para português técnico com termos como "kWh/m²/ano", "L/descarga" e acrônimos (HVAC, BEMS, VRF) gerou subpalavras fragmentadas que afetam a qualidade dos embeddings, especialmente para o modelo multilingual-e5-large.


### 8.2 Impacto do Fine-Tuning na Qualidade das Respostas

O fine-tuning QLoRA foi concluído em **17,4 minutos** na GPU T4, com **loss final de 1,4433** após 3 épocas (84 steps, batch efetivo de 16).

**Resultados das métricas automáticas:**

| Métrica | Modelo Base | Modelo FT | Melhoria |
|---|---|---|---|
| ROUGE-L médio | 0.1615 | 0.2007 | +3,93 pp |
| BERTScore F1 | 0.6782 | 0.7075 | +2,93 pp |

**Resultados por pergunta (ROUGE-L):**

| Pergunta | Base | FT | Δ |
|---|---|---|---|
| Redução de água WEp1 LEED v4.1 | 0.2500 | 0.2901 | +0.0401 |
| Cálculo de potência de pico FV | 0.1013 | 0.1359 | +0.0347 |
| Certificação AQUA-HQE e perfis | 0.1190 | 0.1390 | +0.0200 |
| Parâmetros NBR 16783 reuso vasos | 0.1484 | 0.1489 | +0.0005 |
| EUI alvo escritório Net Zero | 0.2491 | 0.3288 | **+0.0796** |
| BEMS e protocolos | 0.1569 | 0.2119 | +0.0550 |
| QLoRA vs LoRA convencional | 0.1749 | 0.2381 | +0.0632 |
| Zonas bioclimáticas NBR 15575 | 0.0928 | 0.1303 | +0.0375 |
| Crédito EA2 LEED e pontuação | 0.1436 | 0.1576 | +0.0140 |
| Tratamento de águas cinzas | 0.1787 | 0.2268 | +0.0481 |

**Melhorias observadas:**
- O fine-tuning melhorou o ROUGE-L em **todas as 10 perguntas**, sem nenhuma regressão. A maior melhoria foi na pergunta sobre EUI alvo para escritório Net Zero (+0.0796), onde o modelo base produzia respostas genéricas sobre consumo energético e o modelo fine-tunado passou a citar diretamente valores de referência como "≤ 50 kWh/m²/ano" e a relação com ASHRAE 90.1-2016.
- O BERTScore subiu de 0.6782 para 0.7075 (+2,93 pp), indicando que as respostas do modelo fine-tunado são semanticamente mais próximas das referências técnicas, não apenas lexicalmente, mas em conteúdo e significado.
- O modelo fine-tunado passou a usar corretamente siglas e nomenclaturas do domínio (ISPG, PR, EUI, QB1-QB14, WEp1) que o modelo base ou ignorava ou explicava incorretamente.
- Respostas estruturadas com valores numéricos e referências normativas específicas aumentaram significativamente no modelo ajustado.

**Limitações identificadas:**
- A pergunta sobre parâmetros da NBR 16783 teve melhoria mínima (+0.0005), provavelmente porque envolve valores numéricos muito específicos (coliformes ≤ 200 NMP/100 mL, DBO₅ ≤ 30 mg/L) que o modelo ainda confabula com ligeiras variações, o corpus sintético pode não ter reforçado esses valores com repetição suficiente.
- A pergunta sobre zonas bioclimáticas teve o menor ROUGE-L mesmo após o fine-tuning (0.1303), pois exige memorização de 8 zonas com cidades específicas — tarefa que se beneficiaria de mais exemplos de treinamento cobrindo cada zona individualmente.
- Com apenas ~443 exemplos de treino, o modelo generaliza bem para perguntas estruturalmente similares às do corpus, mas pode confabular em perguntas com recortes muito específicos não representados nos pares Q&A.
- O loss final de 1,4433 indica que o modelo ainda tem margem para melhora valores abaixo de 1,0 seriam ideais para um domínio técnico restrito.


### 8.3 Propostas de Melhoria

**Melhoria 1: Expansão do corpus e dos pares Q&A com geração sintética:**
Com mais tempo e recursos, utilizaríamos o próprio modelo (ou a API Claude/GPT-4) para gerar automaticamente 1.000+ pares Q&A a partir dos 10 documentos do corpus, cobrindo perguntas de diferentes tipos (conceitual, numérica, comparativa, aplicada). Isso seria feito com um pipeline de geração → revisão humana → filtragem por qualidade (BLEU score mínimo de 0.3 versus corpus), chegando a 1.000–1.500 pares de qualidade para replicar o nível de qualidade esperado pelo enunciado.

**Melhoria 2: Fine-tuning com feedback humano (RLHF/DPO):**
Após o SFT (Supervised Fine-Tuning) atual, aplicaríamos DPO (Direct Preference Optimization) com pares de respostas rankeadas por especialistas: para cada pergunta, um engenheiro avaliaria qual resposta (A ou B) é mais precisa tecnicamente. O DPO otimiza diretamente o modelo para preferir respostas de maior qualidade sem precisar de um reward model separado, o que seria mais eficiente que PPO para nosso volume de dados.

**Melhoria 3: RAG com re-ranking e metadados filtrados:**
Implementar re-ranking com cross-encoder (ex.: `cross-encoder/ms-marco-MiniLM-L-6-v2`) para refinar os K resultados do ChromaDB antes de incluí-los no prompt. Adicionar filtros por metadados (ex.: recuperar apenas chunks de documentos de `subcategoria: agua` para perguntas sobre reuso) para reduzir a quantidade de contexto irrelevante injetado no prompt.


In [ ]:
# Célula final — Sumário do projeto
ckpt = load_checkpoint()
print("=" * 60)
print("📊 SUMÁRIO FINAL DO PROJETO")
print("=" * 60)
print(f"  Última etapa concluída: {ckpt.get('stage_completed', 0)}/7")
print(f"  Documentos no corpus: {ckpt.get('docs', 0)}")
print(f"  Pares Q&A utilizados: {ckpt.get('qa_pairs', 0)}")
print(f"  Chunks indexados: {ckpt.get('chunks', 0)}")
print(f"  Loss final do treino: {ckpt.get('loss_final', 'N/A')}")
print(f"  ROUGE-L base → FT: {ckpt.get('rouge_l_base', 'N/A')} → {ckpt.get('rouge_l_ft', 'N/A')}")
print(f"  BERTScore base → FT: {ckpt.get('bertscore_base', 'N/A')} → {ckpt.get('bertscore_ft', 'N/A')}")
print("=" * 60)
print("✅ Projeto concluído! Arquivos gerados:")
for p in sorted(BASE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {str(p)}")


📊 SUMÁRIO FINAL DO PROJETO
  Última etapa concluída: 7/7
  Documentos no corpus: 0
  Pares Q&A utilizados: 0
  Chunks indexados: 0
  Loss final do treino: N/A
  ROUGE-L base → FT: N/A → N/A
  BERTScore base → FT: N/A → N/A
✅ Projeto concluído! Arquivos gerados:
  /content/projeto_edificios_verdes/GUIA_PASSO_A_PASSO.md
  /content/projeto_edificios_verdes/README.md
  /content/projeto_edificios_verdes/chatbot/chatbot_app.py
  /content/projeto_edificios_verdes/checkpoint.json
  /content/projeto_edificios_verdes/chroma_db/9d1e37e4-a16f-4944-84ba-f0dc5bcb822b/data_level0.bin
  /content/projeto_edificios_verdes/chroma_db/9d1e37e4-a16f-4944-84ba-f0dc5bcb822b/header.bin
  /content/projeto_edificios_verdes/chroma_db/9d1e37e4-a16f-4944-84ba-f0dc5bcb822b/length.bin
  /content/projeto_edificios_verdes/chroma_db/9d1e37e4-a16f-4944-84ba-f0dc5bcb822b/link_lists.bin
  /content/projeto_edificios_verdes/chroma_db/chroma.sqlite3
  /content/projeto_edificios_verdes/corpus/doc_01_leed_eficiencia_hidrica.txt